# Zalo 2026 — wheels dùng chung

Tải sẵn mọi thư viện cài thêm (ngoài image Kaggle) thành file wheel trong `/kaggle/working/wheels/`, rồi **tự kiểm tra**: cài lại từ chính các wheel đó (không mạng) và chạy vLLM với model nhỏ trên GPU. Kết quả ghi `wheels/SMOKE.json`, phiên bản ghi `wheels/versions.json`.

Hai danh sách:
- `requirements.txt`: cài vào **môi trường gốc** của Kaggle (không được kéo torch mới).
- `requirements-vllm.txt`: cài vào **venv riêng** `/tmp/venv_vllm`. `pip install vllm` vào môi trường gốc sẽ thay torch (2.13+cu130 trên image 2026-10) và làm hỏng torchaudio → ASR transformers chết, nên vLLM luôn ở venv.

**Dùng:** chạy notebook này một lần (GPU T4 x2, Internet On, *Save & Run All*). Ở notebook khác: *Add Input → Your Work → Notebooks →* `zalo2026-wheels`; cell cài đặt tự tìm `/kaggle/input/**/wheels/` và cài offline, lỗi thì cài từ mạng.

**Thêm thư viện mới:** thêm một dòng vào đúng danh sách → chạy lại notebook (version mới) → `SMOKE.json` phải OK → notebook dùng nó lấy version output mới. Settings → Environment nên chọn *Pin to original environment*; image đổi thì build lại.

In [ ]:
%%writefile /kaggle/working/requirements.txt
# Cài vào MÔI TRƯỜNG GỐC Kaggle. Không thêm gói kéo theo torch/CUDA mới. Mỗi dòng ghi notebook nào dùng.
python-calamine     # đọc xlsx nhanh — datamate
sherpa-onnx         # ASR gipformer — lifemate
soundfile           # đọc wav — lifemate
kagglehub           # tự tải dataset khi chưa Add Input — cả 2
uv                  # tạo venv vLLM nhanh — cả 2
peft                # LoRA — lifemate (đợt 2)
bitsandbytes        # 4-bit — lifemate (đợt 2), model lớn datamate

In [ ]:
%%writefile /kaggle/working/requirements-vllm.txt
# Cài vào VENV RIÊNG /tmp/venv_vllm (worker LLM chạy bằng python của venv). Ghim phiên bản khi đã biết bản chạy được trên T4.
vllm==0.32.0        # LLM nhanh — lifemate_agent, datamate_helper_agent (chạy được trên T4, smoke 2026-10-29)
wrapt               # sitecustomize của image Kaggle cần trong venv
pandas              # executor + loader DataMate chạy trong worker
pyarrow
openpyxl
python-calamine

In [ ]:
import os, sys, json, glob, subprocess, platform, time, shutil
from pathlib import Path
W = Path("/kaggle/working/wheels"); W.mkdir(parents=True, exist_ok=True)
REQS = ["/kaggle/working/requirements.txt", "/kaggle/working/requirements-vllm.txt"]

def sh(cmd):
    print("$", cmd); r = subprocess.run(cmd, shell=True, capture_output=True, text=True); print((r.stdout + r.stderr)[-4000:]); return r.returncode

def ver(m):
    try:
        return __import__(m).__version__
    except Exception:
        return None

PREINSTALLED = {m: ver(m) for m in ("torch", "torchaudio", "transformers", "numpy", "pandas")}  # before anything is installed
print(PREINSTALLED)
t = time.time()
rc = 0
for req in REQS:
    rc |= sh(f"{sys.executable} -m pip download --prefer-binary -r {req} -d {W}")
# packages published only as source: turn them into wheels so the offline install never compiles
for sdist in glob.glob(f"{W}/*.tar.gz") + glob.glob(f"{W}/*.zip"):
    if sh(f"{sys.executable} -m pip wheel --no-deps -w {W} {sdist}") == 0:
        os.remove(sdist)
for req in REQS:
    shutil.copy(req, W / Path(req).name)
print("download rc", rc, f"{(time.time() - t) / 60:.1f} min,", len(list(W.glob("*.whl"))), "wheels,",
      round(sum(p.stat().st_size for p in W.iterdir()) / 2**30, 2), "GB")
assert rc == 0, "pip download failed"

In [ ]:
# Record what was built and on which image
versions = {"built": time.strftime("%Y-%m-%d %H:%M:%S %z"), "python": platform.python_version(), "preinstalled": PREINSTALLED,
            "wheels": sorted(p.name for p in W.glob("*.whl"))}
try:
    versions["gpu"] = subprocess.run(["nvidia-smi", "--query-gpu=name,driver_version", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
except Exception:
    pass
(W / "versions.json").write_text(json.dumps(versions, indent=1, ensure_ascii=False))
print(json.dumps({k: v for k, v in versions.items() if k != "wheels"}, indent=1))

In [ ]:
# Smoke test, exactly as the experiment notebooks install: base list into the Kaggle env, vLLM list into its own venv
smoke = {"base_install_rc": sh(f"{sys.executable} -m pip install -q --no-index --find-links {W} -r {REQS[0]}")}
VENV = Path("/tmp/venv_vllm")  # not under /kaggle/working: it would be saved as output (GBs)
sh(f"{sys.executable} -m uv venv -q {VENV} --python {sys.executable}")
VPY = str(VENV / "bin" / "python")
smoke["vllm_venv_install_rc"] = sh(f"{sys.executable} -m uv pip install -q --python {VPY} --no-index --find-links {W} -r {REQS[1]}")
VLLM_CHECK = """
from vllm import LLM, SamplingParams
llm = LLM(model="Qwen/Qwen3-0.6B", dtype="half", max_model_len=2048, gpu_memory_utilization=0.5, seed=0)
out = llm.generate(["Xin chào, 1 + 1 bằng mấy?"], SamplingParams(temperature=0, max_tokens=16))
print("OUT:", out[0].outputs[0].text)
"""
checks = {
    "vllm_t4 (venv)": (VPY, VLLM_CHECK),
    "base torch unchanged": (sys.executable, f"import torch; print(torch.__version__); assert torch.__version__ == {PREINSTALLED['torch']!r}"),
    "base transformers pipeline (ASR path)": (sys.executable, "from transformers import pipeline; print('pipeline ok')"),
    "python_calamine": (sys.executable, "import python_calamine; print('ok')"),
    "sherpa_onnx": (sys.executable, "import sherpa_onnx; print(sherpa_onnx.__version__)"),
    "soundfile": (sys.executable, "import soundfile; print(soundfile.__version__)"),
    "kagglehub": (sys.executable, "import kagglehub; print(kagglehub.__version__)"),
    "peft": (sys.executable, "import peft; print(peft.__version__)"),
    "bitsandbytes": (sys.executable, "import bitsandbytes, torch; print(bitsandbytes.__version__, torch.cuda.is_available())"),
    "venv pandas stack": (VPY, "import pandas, pyarrow, openpyxl, python_calamine; print(pandas.__version__)"),
}
for name, (py, src) in checks.items():
    r = subprocess.run([py, "-c", src], capture_output=True, text=True, timeout=1800)
    smoke[name] = {"ok": r.returncode == 0, "tail": (r.stdout + r.stderr)[-1500:]}
    print(f"{name}: {'OK' if r.returncode == 0 else 'FAIL'}")
    print(smoke[name]["tail"][-600:])
(W / "SMOKE.json").write_text(json.dumps(smoke, indent=1, ensure_ascii=False))
print("SMOKE:", {k: (v["ok"] if isinstance(v, dict) else v) for k, v in smoke.items()})